# VQE - Variational Quantum Eigensolver
### Ground-state energies of molecules and spin chains with NVIDIA CUDA-Q
*Notebook 1 of 8 - CUDA-Q Quantum Algorithm Benchmarking Suite*

This notebook finds the lowest energy (ground state) of a Hamiltonian with the Variational Quantum Eigensolver: a parameterised quantum circuit prepares trial states, CUDA-Q measures their energy and a classical optimizer lowers it. The result is compared with an exact classical reference, and time and memory are benchmarked on the selected backend.

**How to use this notebook:** run all cells from top to bottom (*Kernel -> Restart & Run All*).
Change parameters **only in Section 3**; every later cell reads them from there. The default
configuration runs immediately on a laptop CPU and much faster on an NVIDIA GPU.

| Section | Content |
|---|---|
| 1 | Problem statement and mathematical explanation |
| 2 | Import libraries (with installation instructions) |
| 3 | Input parameters - **the only cell you need to edit** |
| 4 | Example inputs |
| 5 | Benchmark utilities and backend initialization |
| 6 | Expected output (computed *before* running the quantum circuit) |
| 7 | Quantum circuit generation and visualization |
| 8 | Execution with live memory monitoring |
| 9 | Results and visualization |
| 10 | Final benchmark report |

---
## Section 1 - Problem statement and mathematical explanation

### What problem does the algorithm solve?
Find the **ground-state energy** $E_0$, the smallest eigenvalue of a Hamiltonian $H$ acting on $N$ qubits.
Every Hamiltonian used here is a weighted sum of Pauli strings,

$$H = \sum_k c_k P_k,\qquad P_k \in \{I, X, Y, Z\}^{\otimes N}.$$

### Why is this problem important?
Ground-state energies determine chemical bond energies, reaction rates, material properties and magnetism.
Molecules map to qubit Hamiltonians via the Jordan-Wigner transformation; spin models (like the Ising chain below)
describe magnets and are standard benchmarks. VQE is the most widely used algorithm for today's noisy quantum
hardware because it uses short circuits and lets a classical computer do the optimisation.

### What is the classical approach?
*Exact diagonalisation* of the $2^N\times 2^N$ matrix: memory and time grow exponentially (N = 20 already needs a
million-dimensional sparse eigenproblem). Approximate methods (Hartree-Fock, coupled cluster, DMRG, quantum Monte
Carlo) are efficient for many systems but fail for strongly correlated ones. Special models have exact solutions:
the 1-D transverse-field Ising chain used by default can be solved for any $N$ with free fermions, which gives an
exact reference even for 100 qubits.

### How does the quantum algorithm solve it?
The **variational principle** guarantees that for every normalised state

$$E(\boldsymbol\theta) = \langle\psi(\boldsymbol\theta)|H|\psi(\boldsymbol\theta)\rangle \;\ge\; E_0 .$$

1. A parameterised circuit (the **ansatz**) $U(\boldsymbol\theta)$ prepares $|\psi(\boldsymbol\theta)\rangle = U(\boldsymbol\theta)|0\rangle^{\otimes N}$.
2. CUDA-Q computes $E(\boldsymbol\theta) = \sum_k c_k\langle P_k\rangle$ with `cudaq.observe` (exactly on simulators, from
   measurements in the eigenbasis of each $P_k$ on hardware).
3. A classical optimizer (COBYLA, Nelder-Mead, ...) changes $\boldsymbol\theta$ to lower $E(\boldsymbol\theta)$; repeat.

Ansatz options:
* **Hardware-efficient** (`"hea"`): layers of $R_Y(\theta)$ rotations on every qubit followed by a ladder of CNOTs;
  $(L+1)N$ parameters for $L$ layers. Works for any Hamiltonian.
* **UCC double excitation** (`"ucc"`, H$_2$ only): starting from the Hartree-Fock state $|1100\rangle$,
  $|\psi(\theta)\rangle = e^{i\theta X_0X_1X_2Y_3}|1100\rangle = \cos\theta|1100\rangle - \sin\theta|0011\rangle$ -
  one parameter, chemically motivated.

### Default Hamiltonian: transverse-field Ising chain (TFIM)
$$H = -J\sum_{i=0}^{N-2} Z_iZ_{i+1} \;-\; h\sum_{i=0}^{N-1} X_i \quad\text{(open chain)}.$$
Exact $E_0$: map to free fermions (Jordan-Wigner); with the $2N\times 2N$ real antisymmetric matrix $A$ of the
resulting Majorana quadratic form, $E_0 = -\tfrac12\sum|\lambda(iA)|$ - computed in Section 4 for any $N$.

### Example molecule: H$_2$
Hydrogen at bond length 0.7414 &Aring;, STO-3G basis, Jordan-Wigner mapping: 4 qubits, 15 Pauli terms
(coefficients in Section 4). Exact (full CI) ground energy: $-1.137284$ Hartree; **chemical accuracy** = 1.6 mHa.

### Measurement
On hardware each Pauli term is measured after basis rotations ($H$ for $X$, $S^\dagger H$ for $Y$); the energy is a
weighted average of $\pm1$ outcomes. At the end the optimised state is also **sampled** to show which bitstrings
dominate it. For the ordered Ising chain ($h < J$) the exact ground state is close to the symmetric superposition of
the two ferromagnetic states $|00\ldots0\rangle$ and $|11\ldots1\rangle$; a VQE run usually settles on *one* of them
("symmetry breaking"), which costs only an energy exponentially small in $N$.

### What is the expected solution?
The energy curve decreases towards $E_0$ **from above** (variational principle). The remaining gap measures how well
the ansatz and optimizer can represent the true ground state: H$_2$ with the UCC ansatz reaches chemical accuracy;
the 15-qubit Ising chain with 2 hardware-efficient layers typically ends within a fraction of a percent of $E_0$.

---
## Section 2 - Import libraries

All quantum code uses **NVIDIA CUDA-Q** (`cudaq`). The other libraries are standard scientific Python.

**Installation** (Linux x86_64/aarch64 with or without an NVIDIA GPU, or macOS on Apple silicon for CPU-only simulation):

```bash
python3 -m venv cudaq-env && source cudaq-env/bin/activate     # Python >= 3.11
pip install --upgrade pip
pip install cudaq numpy scipy matplotlib psutil nvidia-ml-py jupyter
```

* `pip install cudaq` detects your CUDA driver and installs the matching CUDA-Q build (CUDA 12.x or 13.x).
  A GPU is **not** required: CPU simulators are always available.
* `nvidia-ml-py` provides NVML bindings used for GPU memory monitoring (optional on CPU-only machines).
* Multi-GPU / multi-node runs additionally need MPI; see the README of this repository.

The next cell imports everything, prints the versions found and warns about incompatible versions.

In [ ]:
# ---------------------------------------------------------------------------------
# Section 2 - Import libraries and check versions
# ---------------------------------------------------------------------------------
import os, sys, re, json, math, time, socket, platform      # standard library: files, timing, system info
import threading, subprocess, tempfile, datetime            # standard library: live memory monitor + report
import warnings                                             # used to hide two harmless CUDA-Q notices

# CUDA-Q 0.16 prints two informational warnings when imported: a "preview" notice for the
# bundled cudaq-logical package and a notice that sample/observe will change in a FUTURE
# release. Neither affects this notebook, so exactly these two messages are hidden.
warnings.filterwarnings("ignore", message=".*cudaq-logical is in preview.*")
warnings.filterwarnings("ignore", message=".*algorithmic primitives will change.*")

_missing = []                                               # collect missing packages for one clear message
try:
    import numpy as np                                      # arrays, linear algebra, random numbers
except ImportError:
    _missing.append("numpy")
try:
    import matplotlib                                       # plotting library
    import matplotlib.pyplot as plt                         # pyplot interface for figures
except ImportError:
    _missing.append("matplotlib")
try:
    import scipy                                            # scientific computing (optimizers, linear algebra)
    from scipy.optimize import minimize               # classical optimizer
    import scipy.sparse as sparse                     # sparse matrices for exact diagonalisation
    import scipy.sparse.linalg as sparse_linalg       # sparse eigenvalue solver (eigsh)
except ImportError:
    _missing.append("scipy")
try:
    import psutil                                           # host RAM monitoring
except ImportError:
    _missing.append("psutil")
try:
    import cudaq                                            # NVIDIA CUDA-Q: kernels, simulators, QPU targets
except ImportError:
    _missing.append("cudaq")
if _missing:
    raise ImportError(f"Missing packages: {_missing}. Install them with:\n"
                      f"    pip install {' '.join(_missing)} nvidia-ml-py\n"
                      "CUDA-Q needs Linux (x86_64/aarch64) or macOS on Apple silicon (CPU only); see README.md.")

try:
    import pynvml                                           # NVIDIA Management Library bindings (GPU memory)
    HAVE_NVML = True
except ImportError:                                         # optional: only needed for GPU monitoring
    pynvml, HAVE_NVML = None, False

try:                                                        # live display inside Jupyter
    from IPython import get_ipython
    from IPython.display import display, Pretty
    _IN_JUPYTER = get_ipython() is not None and "IPKernelApp" in get_ipython().config
except Exception:                                           # running as a plain Python script
    display, Pretty, _IN_JUPYTER = None, None, False

# ---- Version report and compatibility check ----------------------------------------
_match = re.search(r"(\d+)\.(\d+)\.(\d+)", cudaq.__version__)
CUDAQ_VERSION = ".".join(_match.groups()) if _match else cudaq.__version__
_cudaq_tuple = tuple(int(x) for x in _match.groups()) if _match else (0, 0, 0)
print(f"Python      : {platform.python_version()} ({platform.system()} {platform.machine()})")
print(f"CUDA-Q      : {CUDAQ_VERSION}")
print(f"NumPy       : {np.__version__}")
print(f"SciPy       : {scipy.__version__}")
print(f"Matplotlib  : {matplotlib.__version__}")
print(f"psutil      : {psutil.__version__}")
print(f"nvidia-ml-py: {'installed' if HAVE_NVML else 'NOT installed (GPU memory monitoring disabled: pip install nvidia-ml-py)'}")
print(f"GPUs visible to CUDA-Q: {cudaq.num_available_gpus()}")
if sys.version_info < (3, 11):
    print("WARNING: CUDA-Q 0.16 wheels require Python >= 3.11.")
if _cudaq_tuple < (0, 12, 0):
    print("WARNING: this notebook was written and tested with CUDA-Q 0.16.0; versions older than 0.12 "
          "use a different operator API and are not supported. Upgrade with: pip install -U cudaq")
elif _cudaq_tuple != (0, 16, 0):
    print("Note: tested with CUDA-Q 0.16.0. Other versions should work; if an API error appears, "
          "compare with the README's 'Known limitations'.")

# ---------------------------------------------------------------------------------
# Plot style used by every figure: colour-blind-safe palette, hairline grid, one y-axis per chart
# ---------------------------------------------------------------------------------
# Categorical colours in a FIXED order (validated for colour-vision deficiency).
SERIES_COLORS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948"]
BLUE, ORANGE, AQUA = SERIES_COLORS[0], SERIES_COLORS[1], SERIES_COLORS[2]
INK, INK_SECONDARY, INK_MUTED = "#0b0b0b", "#52514e", "#898781"   # text colours
GRID_COLOR, AXIS_COLOR, SURFACE = "#e1e0d9", "#c3c2b7", "#fcfcfb"  # chart chrome
GOOD, CRITICAL = "#0ca30c", "#d03b3b"                                # reserved status colours
SEQUENTIAL_BLUE = matplotlib.colors.LinearSegmentedColormap.from_list(
    "seq_blue", ["#cde2fb", "#86b6ef", "#3987e5", "#1c5cab", "#0d366b"])          # magnitude
DIVERGING_BLUE_RED = matplotlib.colors.LinearSegmentedColormap.from_list(
    "div_blue_red", ["#184f95", "#6da7ec", "#f0efec", "#ec8a89", "#a32b2b"])      # signed values


def apply_plot_style():
    """Consistent, quiet chart style for every figure in this notebook."""
    plt.rcParams.update({
        "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
        "axes.edgecolor": AXIS_COLOR, "axes.labelcolor": INK_SECONDARY, "axes.titlecolor": INK,
        "text.color": INK, "xtick.color": AXIS_COLOR, "ytick.color": AXIS_COLOR,
        "xtick.labelcolor": INK_SECONDARY, "ytick.labelcolor": INK_SECONDARY,
        "axes.grid": True, "grid.color": GRID_COLOR, "grid.linewidth": 0.8, "grid.linestyle": "-",
        "axes.axisbelow": True, "axes.spines.top": False, "axes.spines.right": False,
        "axes.prop_cycle": matplotlib.cycler(color=SERIES_COLORS),
        "lines.linewidth": 2.0, "lines.solid_capstyle": "round", "lines.markersize": 6,
        "legend.frameon": False, "font.size": 10, "axes.titlesize": 11, "figure.dpi": 100,
    })


apply_plot_style()

---
## Section 3 - Input parameters

This is the **only cell you need to edit**. Every later cell reads these variables.

In [ ]:
# =====================================================================================
# Section 3 - INPUT PARAMETERS  (edit only this cell, then re-run the notebook)
# =====================================================================================

# ---- Problem definition: the Hamiltonian ------------------------------------------------
HAMILTONIAN = "tfim"             # "tfim" (Ising chain, any N) | "H2" (hydrogen molecule, 4 qubits) | "custom"
N_QUBITS = 15                    # qubits for "tfim" / "custom" (H2 always uses 4); try 20 / 25 / 30 on GPUs
TFIM_J = 1.0                     # "tfim": nearest-neighbour ZZ coupling J
TFIM_H = 0.5                     # "tfim": transverse field h (h = J is the critical point, hardest for VQE)
CUSTOM_HAMILTONIAN = [(1.0, "ZZI"), (1.0, "IZZ"), (0.7, "XII"), (0.7, "IXI"), (0.7, "IIX")]
                                 # "custom": list of (coefficient, Pauli string); character i acts on qubit i

# ---- Ansatz ------------------------------------------------------------------------------
ANSATZ = "auto"                  # "auto" (UCC for H2, hardware-efficient otherwise) | "hea" | "ucc"
ANSATZ_LAYERS = 2                # hardware-efficient layers L -> (L+1)*N parameters
INITIAL_PARAMETER_SCALE = 0.1    # initial angles ~ Normal(0, scale) (UCC starts at 0 = Hartree-Fock)

# ---- Optimizer -----------------------------------------------------------------------------
OPTIMIZER = "COBYLA"             # "COBYLA" | "Nelder-Mead" | "Powell" | "L-BFGS-B"
MAX_EVALUATIONS = 300            # budget of energy evaluations (= quantum circuit runs)
OBJECTIVE_SHOTS = None           # None: exact <H> (simulators); integer: estimate <H> from shots (hardware-like)
REFERENCE_MAX_QUBITS = 18        # exact diagonalisation for "custom"/"H2" only up to this N

# ---- Backend and simulation method ---------------------------------------------------
BACKEND = "auto"                 # "auto" | "cpu" | "gpu" | "multi-gpu" | "hardware"
SIMULATION_METHOD = "statevector"   # this notebook supports: "statevector" | "mps" | "tensornet" | "density_matrix"
GPU_PRECISION = "fp64"            # "fp64" (16 B/amplitude) | "fp32" (8 B/amplitude, half the memory)
HARDWARE_PROVIDER = "quantinuum"  # only for BACKEND="hardware": "quantinuum", "ionq", "oqc", "anyon", "infleqtion", ...
HARDWARE_EMULATE = True           # True: local emulation (no account, no cost); False: real QPU (credentials needed)
HARDWARE_MACHINE = None           # provider device name, e.g. "H2-1E"; None = provider default

# ---- Shots and reproducibility --------------------------------------------------------
SHOTS = 4000                       # measurement repetitions; statistical error ~ 1/sqrt(SHOTS)
SEED = 42                         # fixes NumPy and CUDA-Q random numbers -> reproducible results

# ---- Memory monitoring ----------------------------------------------------------------
LIVE_MEMORY_DISPLAY = True        # live RAM / GPU memory line while the execution cell runs
MEMORY_SAMPLE_INTERVAL_S = 0.25   # sampling period in seconds (smaller = finer timeline)
MEMORY_SAFETY_FRACTION = 0.8      # refuse runs whose state needs more than 80% of free memory

# ---- Circuit drawing, export and report ---------------------------------------------------
DRAW_QUBITS = 10                  # qubits shown in the circuit drawing (maximum 10)
EXPORT_CIRCUIT = True             # save drawing + full circuit to EXPORT_DIR
EXPORT_FORMATS = ["txt", "png", "qir"]   # "txt", "png" (drawing), "qir", "qasm" (full circuit)
EXPORT_DIR = "exports/vqe"     # output folder (created automatically)
SAVE_REPORT = True                # save the final benchmark report (JSON) and figure (PNG)

#### What every parameter means and what happens when you change it

| Parameter | Meaning | Effect of changing it |
|---|---|---|
| `HAMILTONIAN` | Which problem: Ising chain, H$_2$ molecule or your own Pauli sum. | Selects the Hamiltonian and the exact reference: free fermions (TFIM, any N), diagonalisation (H$_2$, custom up to `REFERENCE_MAX_QUBITS`). |
| `N_QUBITS` | Qubits for `"tfim"`/`"custom"`. | State memory doubles per qubit; the number of TFIM terms grows like 2N and the number of parameters like (L+1)N, so more optimizer evaluations are needed. |
| `TFIM_J`, `TFIM_H` | Coupling and transverse field. | $h < J$: ordered (ferromagnet, easy); $h \approx J$: critical, long-range entanglement, hardest for shallow circuits; $h > J$: paramagnet. |
| `CUSTOM_HAMILTONIAN` | `(coefficient, "PAULIS")` pairs. | Every string must have exactly `N_QUBITS` characters from I, X, Y, Z; coefficients must be real (H must be Hermitian). |
| `ANSATZ` / `ANSATZ_LAYERS` | Trial-state circuit and its depth. | More layers = more expressive (lower energy reachable) but more parameters (slower optimisation) and deeper circuits (more noise on hardware). |
| `INITIAL_PARAMETER_SCALE` | Spread of random initial angles. | Small values start close to $|00\ldots0\rangle$; large values start from a random state, which can trap the optimizer. |
| `OPTIMIZER` / `MAX_EVALUATIONS` | Classical optimizer and its budget. | Run time is proportional to the budget. COBYLA is robust; L-BFGS-B uses finite-difference gradients (N_params + 1 runs per gradient). |
| `OBJECTIVE_SHOTS` | Exact or sampled energies. | Shot noise ~ 1/sqrt(shots) makes the optimisation noisy, as on real hardware. |
| `REFERENCE_MAX_QUBITS` | Size limit of the exact-diagonalisation reference. | Sparse diagonalisation memory/time grows like 2^N. |
| `BACKEND` | Where the circuit runs: `"auto"` (NVIDIA GPU if CUDA-Q sees one, otherwise CPU), `"cpu"`, `"gpu"`, `"multi-gpu"`, `"hardware"`. | GPU state-vector simulation is typically 10-1000x faster than CPU for N >= 20 and is limited by GPU memory. `"multi-gpu"` pools the memory of several GPUs (launch with MPI, see README). `"hardware"` sends the circuit to a quantum computer provider (sampling only). |
| `SIMULATION_METHOD` | How the quantum state is stored (the allowed values for this notebook are listed in the input cell). `"statevector"`: all 2^N amplitudes, exact. `"mps"`: matrix product state, compressed (GPU). `"tensornet"`: exact tensor-network contraction (GPU). `"density_matrix"`: 4^N entries, supports noise (CPU). | Changes memory and run time: state vector needs 16 x 2^N bytes (fp64), density matrix 16 x 4^N bytes, MPS grows with entanglement instead of with N. An unsupported combination stops with an explanation. |
| `GPU_PRECISION` | `"fp64"` (16 bytes per amplitude) or `"fp32"` (8 bytes). GPU targets only; CPU targets always use fp64. | fp32 halves memory (one more qubit fits) and is faster, with ~1e-7 relative precision instead of ~1e-16. |
| `HARDWARE_PROVIDER` / `HARDWARE_EMULATE` / `HARDWARE_MACHINE` | Used only when `BACKEND="hardware"`: provider target (`"quantinuum"`, `"ionq"`, `"oqc"`, `"anyon"`, `"infleqtion"`, ...), local emulation switch, device name. | `HARDWARE_EMULATE=True` runs the provider's compilation flow locally with no account and no cost. `False` submits real jobs: needs credentials, may cost money, waits in a queue. |
| `SHOTS` | Number of times the circuit is executed and measured. | Statistical error of every measured probability shrinks like 1/sqrt(SHOTS). On simulators extra shots are cheap; on hardware cost grows linearly. |
| `SEED` | Random seed for NumPy and CUDA-Q's shot sampler. | Same seed gives identical results on re-run; change it to see statistical fluctuations. |
| `LIVE_MEMORY_DISPLAY` / `MEMORY_SAMPLE_INTERVAL_S` | Live RAM/GPU memory line during execution; sampling period in seconds. | Smaller intervals give a finer memory timeline; peaks shorter than the interval can be missed. |
| `MEMORY_SAFETY_FRACTION` | The run is refused if the quantum state would need more than this fraction of the free memory. | Raise it (e.g. 0.95) if you know the run fits; lower it to protect other work on the machine. |
| `DRAW_QUBITS` | Number of qubits used for the circuit **drawing** (at most 10). | Only changes the picture; execution always uses the full number of qubits. |
| `EXPORT_CIRCUIT` / `EXPORT_FORMATS` / `EXPORT_DIR` / `SAVE_REPORT` | Save the drawing (`"txt"`, `"png"`), the full circuit (`"qir"`, `"qasm"`), the report (JSON) and final figure (PNG) to `EXPORT_DIR`. | Set `EXPORT_CIRCUIT=False` / `SAVE_REPORT=False` to write nothing to disk. |

---
## Section 4 - Example inputs

**Default example (runs as is):** transverse-field Ising chain with `N_QUBITS = 15`, `TFIM_J = 1.0`, `TFIM_H = 0.5`
and a 2-layer hardware-efficient ansatz (45 parameters).

**Molecular example - hydrogen molecule H$_2$:** set `HAMILTONIAN = "H2"` (4 qubits, UCC ansatz, converges in about
20 evaluations). Its 15 Pauli terms (bond length 0.7414 &Aring;, STO-3G basis, Jordan-Wigner, spin-orbitals interleaved)
are listed in the cell below; the constant term includes the nuclear repulsion energy.

| Goal | Setting in Section 3 |
|---|---|
| Molecule (chemistry) | `HAMILTONIAN = "H2"` |
| Critical Ising chain (harder) | `TFIM_H = 1.0`, `ANSATZ_LAYERS = 4`, `MAX_EVALUATIONS = 800` |
| Your own Hamiltonian | `HAMILTONIAN = "custom"`, `N_QUBITS = 3`, `CUSTOM_HAMILTONIAN = [(1.0, "ZZI"), (0.5, "XIX")]` |
| GPU scaling study | `N_QUBITS = 20 / 25 / 28`, `BACKEND = "gpu"` |

In [ ]:
# ---------------------------------------------------------------------------------
# Section 4 - Build the Hamiltonian, validate it and compute the exact reference energy
# ---------------------------------------------------------------------------------
H2_TERMS = [  # (coefficient in Hartree, Pauli string on qubits 0..3): H2, 0.7414 Angstrom, STO-3G, Jordan-Wigner
    (-0.0970662681676282, "IIII"), (0.17141282644776884, "ZIII"), (0.17141282644776884, "IZII"),
    (-0.22343153690813597, "IIZI"), (-0.22343153690813597, "IIIZ"), (0.16868898170361213, "ZZII"),
    (0.12062523483390425, "ZIZI"), (0.16592785033770352, "ZIIZ"), (0.16592785033770352, "IZZI"),
    (0.12062523483390425, "IZIZ"), (0.17441287612261608, "IIZZ"), (-0.0453026155037993, "XXYY"),
    (0.0453026155037993, "XYYX"), (0.0453026155037993, "YXXY"), (-0.0453026155037993, "YYXX")]
H2_FCI_LITERATURE = -1.137283834                     # Hartree (full configuration interaction, same basis)

kind = str(HAMILTONIAN).strip().lower()
if kind == "h2":
    n_qubits, pauli_terms = 4, list(H2_TERMS)
elif kind == "tfim":
    if not isinstance(N_QUBITS, (int, np.integer)) or N_QUBITS < 2:
        raise ValueError("N_QUBITS must be an integer >= 2 for the Ising chain.")
    n_qubits = int(N_QUBITS)
    pauli_terms = []
    for i in range(n_qubits - 1):                     # -J Z_i Z_{i+1}
        word = ["I"] * n_qubits; word[i] = word[i + 1] = "Z"
        pauli_terms.append((-float(TFIM_J), "".join(word)))
    for i in range(n_qubits):                         # -h X_i
        word = ["I"] * n_qubits; word[i] = "X"
        pauli_terms.append((-float(TFIM_H), "".join(word)))
elif kind == "custom":
    n_qubits = int(N_QUBITS)
    pauli_terms = [(float(np.real(c)), str(w).upper()) for c, w in CUSTOM_HAMILTONIAN]
    for c, w in CUSTOM_HAMILTONIAN:
        if abs(np.imag(c)) > 1e-12:
            raise ValueError(f"Coefficient {c} of {w} is complex: the Hamiltonian must be Hermitian (real coefficients).")
else:
    raise ValueError("HAMILTONIAN must be 'tfim', 'H2' or 'custom'.")
for c, w in pauli_terms:
    if len(w) != n_qubits or set(w) - set("IXYZ"):
        raise ValueError(f"Pauli string {w!r} must have exactly {n_qubits} characters from I, X, Y, Z.")

ansatz = str(ANSATZ).strip().lower()
if ansatz == "auto":
    ansatz = "ucc" if kind == "h2" else "hea"
if ansatz == "ucc" and kind != "h2":
    raise ValueError("ANSATZ='ucc' is implemented for the H2 example only; use 'hea' for other Hamiltonians.")
if ansatz not in ("ucc", "hea"):
    raise ValueError("ANSATZ must be 'auto', 'hea' or 'ucc'.")
num_parameters = 1 if ansatz == "ucc" else (int(ANSATZ_LAYERS) + 1) * n_qubits

def build_spin_operator(terms):
    """sum_k c_k P_k as a CUDA-Q SpinOperator (character i of the string acts on qubit i)."""
    operator = None
    for coefficient, word in terms:
        term = coefficient * cudaq.SpinOperator.from_word(word)
        operator = term if operator is None else operator + term
    return operator

_PAULI = {"I": sparse.identity(2, format="csr", dtype=complex),
          "X": sparse.csr_matrix(np.array([[0, 1], [1, 0]], dtype=complex)),
          "Y": sparse.csr_matrix(np.array([[0, -1j], [1j, 0]], dtype=complex)),
          "Z": sparse.csr_matrix(np.array([[1, 0], [0, -1]], dtype=complex))}

def exact_ground_energy_sparse(terms, n):
    """Lowest eigenvalue by sparse exact diagonalisation (independent of CUDA-Q)."""
    matrix = sparse.csr_matrix((2 ** n, 2 ** n), dtype=complex)
    for coefficient, word in terms:
        product = _PAULI[word[0]]
        for ch in word[1:]:
            product = sparse.kron(product, _PAULI[ch], format="csr")
        matrix = matrix + coefficient * product
    if 2 ** n <= 64:
        return float(np.linalg.eigvalsh(matrix.toarray())[0])
    return float(sparse_linalg.eigsh(matrix, k=1, which="SA")[0][0])

def tfim_exact_energy(n, J, h):
    """Exact TFIM ground energy for ANY n via free fermions: E0 = -1/2 sum |eig(iA)|."""
    A = np.zeros((2 * n, 2 * n))
    for j in range(n):                                 # -h X_j = -h i a_j b_j
        A[2 * j, 2 * j + 1], A[2 * j + 1, 2 * j] = -h, h
    for j in range(n - 1):                             # -J Z_j Z_{j+1} = -J i b_j a_{j+1}
        A[2 * j + 1, 2 * j + 2], A[2 * j + 2, 2 * j + 1] = -J, J
    return -0.5 * float(np.sum(np.abs(np.linalg.eigvalsh(1j * A))))

t0 = time.perf_counter()
reference_energy, reference_kind = None, "not available"
if kind == "tfim":
    reference_energy = tfim_exact_energy(n_qubits, float(TFIM_J), float(TFIM_H))
    reference_kind = "exact (free-fermion solution)"
    if n_qubits <= 16:                                 # independent cross-check by diagonalisation
        cross_check = exact_ground_energy_sparse(pauli_terms, n_qubits)
        reference_kind += f"; sparse diagonalisation gives {cross_check:.10f}"
elif n_qubits <= REFERENCE_MAX_QUBITS:
    reference_energy = exact_ground_energy_sparse(pauli_terms, n_qubits)
    reference_kind = "exact (sparse diagonalisation)"
reference_time = time.perf_counter() - t0

hamiltonian = build_spin_operator(pauli_terms)
print(f"Hamiltonian         : {kind.upper()} on {n_qubits} qubits, {len(pauli_terms)} Pauli terms")
for c, w in pauli_terms[:6]:
    print(f"   {c:+.6f} * {w}")
if len(pauli_terms) > 6:
    print(f"   ... ({len(pauli_terms) - 6} more terms)")
print(f"Ansatz              : {'UCC double excitation (1 parameter)' if ansatz == 'ucc' else f'hardware-efficient, {ANSATZ_LAYERS} layers'}"
      f" -> {num_parameters} parameters")
if reference_energy is not None:
    print(f"Reference energy E0 : {reference_energy:.10f}   [{reference_kind}, {reference_time:.2f} s]")
else:
    print(f"Reference energy E0 : not computed (N > REFERENCE_MAX_QUBITS = {REFERENCE_MAX_QUBITS})")
if kind == "h2":
    print(f"Literature FCI      : {H2_FCI_LITERATURE:.9f} Hartree (agrees with the diagonalisation)")

---
## Section 5 - Benchmark utilities and backend initialization

The first cell below defines helper functions shared by all eight notebooks (you do not need to edit it):

* **Backend selection** - translates `BACKEND` + `SIMULATION_METHOD` into a real CUDA-Q target and
  refuses combinations CUDA-Q does not support (for example MPS on a CPU) with an explanation.
* **Memory estimation** - exact formulas for the memory needed by the quantum state.
* **Live memory monitor** - a small separate process samples host RAM (`psutil`) and the memory of
  every NVIDIA GPU (NVML) while the execution cell runs.
* **Timing** - separates circuit construction, compilation and execution using CUDA-Q's built-in tracer.
* **Circuit drawing / export** and the **final benchmark report**.

The second cell activates the selected backend and checks that the problem fits into memory.

| Your choice | CUDA-Q target used | Notes |
|---|---|---|
| `cpu` + `statevector` | `qpp-cpu` | OpenMP state vector, fp64 |
| `cpu` + `density_matrix` | `density-matrix-cpu` | mixed states and noise, fp64, 4^N memory |
| `cpu` + `stabilizer` | `stim` | Clifford circuits + Pauli noise only (noise notebook) |
| `gpu` + `statevector` | `nvidia` (option fp32/fp64) | cuStateVec, one GPU |
| `gpu` + `trajectory` | `nvidia` + noise model | Monte-Carlo noise trajectories (noise notebook) |
| `gpu` + `mps` | `tensornet-mps` | cuTensorNet matrix product state |
| `gpu` + `tensornet` | `tensornet` | cuTensorNet exact contraction |
| `multi-gpu` + `statevector` | `nvidia` (option `mgpu,fp64`) | state vector split over GPUs; needs `mpiexec` |
| `multi-gpu` + `tensornet` | `tensornet` | contraction distributed over GPUs; needs `mpiexec` |
| `hardware` | provider target (`quantinuum`, `ionq`, ...) | sampling only; `emulate=True` runs locally |

In [ ]:
# =====================================================================================
# BENCHMARK & MONITORING UTILITIES  (this cell is identical in all 8 notebooks)
# -------------------------------------------------------------------------------------
# You do not need to edit this cell. It defines helper functions used later:
#   resolve_backend()       -> turns BACKEND + SIMULATION_METHOD into a real CUDA-Q target
#   initialize_backend()    -> calls cudaq.set_target() safely
#   estimate_memory_bytes() -> memory needed to store the quantum state (exact formulas)
#   check_memory()          -> compares that requirement with free RAM / GPU memory
#   MemoryMonitor           -> live RAM + GPU memory monitoring during execution
#   profile_first_launch()  -> splits a kernel launch into construction/compilation/execution
#   measure_gate_time()     -> short calibration used for execution-time estimates
#   show_circuit()          -> prints a <=10-qubit circuit drawing and exports it
#   circuit_statistics()    -> gate counts/depth of the FULL circuit (no simulation needed)
#   show_report()           -> prints and saves the final benchmark report
# =====================================================================================

# ---------------------------------------------------------------------------------
# 1. Backend selection
# ---------------------------------------------------------------------------------
# (device, simulation method) -> (CUDA-Q target name, does the target accept fp32/fp64?)
# Only combinations that CUDA-Q actually ships are listed. Anything else raises a clear
# error instead of silently running on a different simulator.
_TARGET_TABLE = {
    ("cpu", "statevector"): ("qpp-cpu", False),             # OpenMP state vector (always fp64)
    ("cpu", "density_matrix"): ("density-matrix-cpu", False),  # exact mixed states (noise), fp64
    ("cpu", "stabilizer"): ("stim", False),                 # Clifford circuits + Pauli noise only
    ("gpu", "statevector"): ("nvidia", True),               # cuStateVec on one GPU
    ("gpu", "trajectory"): ("nvidia", True),                # cuStateVec + noise = Monte-Carlo trajectories
    ("gpu", "mps"): ("tensornet-mps", True),                # cuTensorNet matrix product state
    ("gpu", "tensornet"): ("tensornet", True),              # cuTensorNet exact tensor-network contraction
    ("multi-gpu", "statevector"): ("nvidia", True),         # one state vector split across GPUs (MPI)
    ("multi-gpu", "tensornet"): ("tensornet", True),        # contraction distributed across GPUs (MPI)
}

# Explanations for combinations CUDA-Q does not support (shown to the user on error).
_UNSUPPORTED_HINTS = {
    ("cpu", "mps"): "CUDA-Q's MPS simulator (target 'tensornet-mps') needs an NVIDIA GPU. "
                    "Use BACKEND='gpu', or SIMULATION_METHOD='statevector' on CPU.",
    ("cpu", "tensornet"): "CUDA-Q's tensor-network simulator (target 'tensornet') needs an NVIDIA GPU.",
    ("cpu", "trajectory"): "Trajectory-based noisy simulation runs on GPUs in CUDA-Q. On CPU use "
                           "'density_matrix' (exact, small N) or 'stabilizer' (Clifford circuits).",
    ("gpu", "density_matrix"): "CUDA-Q's density-matrix simulator is CPU-only ('density-matrix-cpu'). "
                               "On GPUs noisy circuits are simulated with 'trajectory'.",
    ("gpu", "stabilizer"): "The stabilizer simulator ('stim') is CPU-only. Use BACKEND='cpu'.",
    ("multi-gpu", "mps"): "The MPS simulator runs on a single GPU. Use BACKEND='gpu'.",
    ("multi-gpu", "density_matrix"): "The density-matrix simulator is CPU-only. Use BACKEND='cpu'.",
    ("multi-gpu", "stabilizer"): "The stabilizer simulator ('stim') is CPU-only. Use BACKEND='cpu'.",
    ("multi-gpu", "trajectory"): "Use BACKEND='gpu' for trajectory simulation in this notebook.",
}


def mpi_rank_info():
    """Return (rank, number_of_ranks). (0, 1) means a normal single-process run."""
    try:
        if cudaq.mpi.is_initialized():                       # CUDA-Q already started MPI
            return cudaq.mpi.rank(), cudaq.mpi.num_ranks()
    except Exception:                                        # cudaq.mpi may be unavailable
        pass
    # Common launcher environment variables (Open MPI, MPICH/Intel MPI, Slurm)
    for rank_var, size_var in (("OMPI_COMM_WORLD_RANK", "OMPI_COMM_WORLD_SIZE"),
                               ("PMI_RANK", "PMI_SIZE"), ("SLURM_PROCID", "SLURM_NTASKS")):
        if rank_var in os.environ:
            return int(os.environ[rank_var]), int(os.environ.get(size_var, "1"))
    return 0, 1


def resolve_backend(backend, method, precision, allowed_methods, hardware_provider=None,
                    hardware_emulate=True, hardware_machine=None):
    """Translate the user's BACKEND / SIMULATION_METHOD choice into a concrete CUDA-Q target.

    Nothing is changed in CUDA-Q here; we only validate and describe the choice.
    Raises ValueError / RuntimeError with an explanation for unsupported choices.
    """
    backend = str(backend).strip().lower()                   # tolerate "GPU", " cpu " ...
    requested = backend                                      # remember the user's original choice
    method = str(method).strip().lower()
    precision = str(precision).strip().lower()
    num_gpus = cudaq.num_available_gpus()                    # GPUs visible to CUDA-Q (0 on CPU-only machines)

    if backend == "auto":                                    # "auto": use a GPU if present, else the CPU
        backend = "gpu" if num_gpus > 0 else "cpu"
    if backend not in ("cpu", "gpu", "multi-gpu", "hardware"):
        raise ValueError(f"BACKEND must be 'auto', 'cpu', 'gpu', 'multi-gpu' or 'hardware', got {backend!r}")
    if precision not in ("fp32", "fp64"):
        raise ValueError(f"GPU_PRECISION must be 'fp32' or 'fp64', got {precision!r}")

    if backend == "hardware":                                # real QPU (or its local emulator)
        provider = str(hardware_provider).strip().lower()
        if not cudaq.has_target(provider):
            raise ValueError(f"CUDA-Q has no target named {provider!r}. "
                             f"Available targets: {sorted(t.name for t in cudaq.get_targets())}")
        kwargs = {}
        if hardware_emulate:
            kwargs["emulate"] = True                         # run locally, no credentials, no cost
        if hardware_machine:
            kwargs["machine"] = hardware_machine             # e.g. "H2-1E" for Quantinuum
        return {"backend_requested": requested, "device": "hardware", "method": "hardware",
                "target": provider, "kwargs": kwargs, "precision": "n/a", "num_gpus": num_gpus,
                "description": f"{provider} QPU " + ("(LOCAL EMULATION - no credentials, no cost)"
                                                     if hardware_emulate else "(REAL HARDWARE - needs credentials, may cost money)")}

    if method not in allowed_methods:                        # notebook-specific restriction
        raise ValueError(f"SIMULATION_METHOD={method!r} is not supported by this notebook. "
                         f"Choose one of {allowed_methods}.")
    if backend in ("gpu", "multi-gpu") and num_gpus == 0:    # never touch GPU targets without a GPU
        raise RuntimeError(f"BACKEND={backend!r} was requested but CUDA-Q sees no NVIDIA GPU "
                           "(cudaq.num_available_gpus() == 0). Check `nvidia-smi`, the driver and "
                           "the CUDA-Q CUDA variant (see README), or use BACKEND='cpu'.")
    key = (backend, method)
    if key not in _TARGET_TABLE:
        raise ValueError(_UNSUPPORTED_HINTS.get(key, f"Combination {key} is not supported by CUDA-Q."))
    target, takes_precision = _TARGET_TABLE[key]

    rank, size = mpi_rank_info()
    note = ""
    if backend == "multi-gpu" and method == "statevector":
        if size > 1:                                         # launched with mpiexec -> pool all GPUs
            option = f"mgpu,{precision}"
        else:                                                # plain Jupyter process: one rank = one GPU
            option = precision
            note = ("Multi-GPU state-vector pooling needs one MPI rank per GPU (see README: "
                    "`mpiexec -np <GPUs> python notebook.py`). This single process uses ONE GPU.")
    else:
        option = precision if takes_precision else ""
    kwargs = {"option": option} if option else {}
    return {"backend_requested": requested, "device": backend, "method": method, "target": target,
            "kwargs": kwargs, "precision": precision if takes_precision else "fp64",
            "num_gpus": num_gpus, "mpi_ranks": size,
            "description": f"{target}" + (f" (option={option})" if option else "") + (f" - {note}" if note else "")}


def initialize_backend(cfg, seed=None):
    """Activate the CUDA-Q target described by cfg (from resolve_backend) and seed its RNG."""
    if cfg["device"] == "multi-gpu" and cfg.get("mpi_ranks", 1) > 1 and not cudaq.mpi.is_initialized():
        cudaq.mpi.initialize()                               # required before using MPI-based targets
    cudaq.set_target(cfg["target"], **cfg["kwargs"])         # switch simulator / QPU
    if seed is not None:
        cudaq.set_random_seed(int(seed))                     # reproducible shot sampling
    return cudaq.get_target()


# ---------------------------------------------------------------------------------
# 2. Memory requirement estimates and availability checks
# ---------------------------------------------------------------------------------
def format_bytes(num_bytes):
    """Human-readable size. Works for astronomically large Python integers as well."""
    if num_bytes is None:
        return "n/a"
    if num_bytes >= 10**18:                                  # beyond exabytes: use powers of ten
        return f"10^{math.log10(num_bytes):.1f} bytes"
    for unit, scale in (("TB", 1e12), ("GB", 1e9), ("MB", 1e6), ("KB", 1e3)):
        if num_bytes >= scale:
            return f"{num_bytes / scale:.2f} {unit}"
    return f"{int(num_bytes)} B"


def estimate_memory_bytes(n_qubits, method, precision="fp64"):
    """Bytes needed to STORE the simulated quantum state (simulator workspace not included).

    statevector / trajectory : 2^N complex amplitudes (8 B each in fp32, 16 B in fp64)
    density_matrix           : 4^N complex entries (16 B each, fp64)
    mps                      : upper bound N * 2 * chi^2 amplitudes, chi = CUDAQ_MPS_MAX_BOND (default 64)
    stabilizer               : ~2N^2 bits (tiny)
    tensornet / hardware     : not predictable -> None
    """
    bytes_per_amplitude = 8 if precision == "fp32" else 16
    if method in ("statevector", "trajectory"):
        return (2 ** n_qubits) * bytes_per_amplitude
    if method == "density_matrix":
        return (4 ** n_qubits) * 16
    if method == "mps":
        chi = int(os.environ.get("CUDAQ_MPS_MAX_BOND", "64"))
        return n_qubits * 2 * chi * chi * bytes_per_amplitude
    if method == "stabilizer":
        return max(1, (2 * n_qubits * n_qubits) // 8)
    return None


def _nvml_gpu_memory():
    """List of (used_bytes, total_bytes, name) per GPU, or (None, reason) if NVML is unavailable."""
    if not HAVE_NVML:
        return None, "nvidia-ml-py (pynvml) is not installed"
    try:
        pynvml.nvmlInit()
        result = []
        for i in range(pynvml.nvmlDeviceGetCount()):
            handle = pynvml.nvmlDeviceGetHandleByIndex(i)
            info = pynvml.nvmlDeviceGetMemoryInfo(handle)
            name = pynvml.nvmlDeviceGetName(handle)
            result.append((info.used, info.total, name.decode() if isinstance(name, bytes) else name))
        return result, "NVML OK"
    except Exception as exc:                                 # no driver / no GPU
        return None, f"NVML unavailable ({type(exc).__name__}: {exc})"


def check_memory(required_bytes, cfg, safety_fraction=0.8, extra_copies=0):
    """Print required vs available memory and stop (MemoryError) if the run cannot fit.

    extra_copies: additional full copies of the state the notebook makes (e.g. get_state()
    returns a host copy of the state vector), added on top of the simulator's own copy.
    """
    if required_bytes is None:
        print("Memory requirement: cannot be predicted for this method (depends on circuit structure).")
        return None
    host_copy = extra_copies * required_bytes
    if cfg["device"] in ("gpu", "multi-gpu"):
        gpus, status = _nvml_gpu_memory()
        if gpus is None:
            print(f"GPU memory check skipped: {status}. Required (state only): {format_bytes(required_bytes)}")
            return None
        used_gpus = gpus if (cfg["device"] == "multi-gpu" and cfg.get("mpi_ranks", 1) > 1) else gpus[:1]
        available = sum(total - used for used, total, _ in used_gpus)
        where = f"free memory on {len(used_gpus)} GPU(s)"
    else:                                                    # CPU simulators and local emulators use host RAM
        available = psutil.virtual_memory().available
        where = "available host RAM"
        required_bytes = required_bytes + host_copy
        host_copy = 0
    print(f"Estimated memory for the quantum state : {format_bytes(required_bytes)}")
    if host_copy:
        print(f"Extra host RAM for state copies        : {format_bytes(host_copy)}")
    print(f"{where:<39}: {format_bytes(available)}  (safety limit {safety_fraction:.0%})")
    if required_bytes > safety_fraction * available:
        raise MemoryError(f"The state needs {format_bytes(required_bytes)} but only "
                          f"{format_bytes(available)} is {where.split(' on ')[0]} (limit {safety_fraction:.0%}). "
                          "Reduce the number of qubits, use fp32, an MPS/tensor-network method, more GPUs, "
                          "or raise MEMORY_SAFETY_FRACTION if you know the run fits.")
    print("Memory check: OK - the state fits within the safety limit.")
    return available


# ---------------------------------------------------------------------------------
# 3. Live memory monitor
# ---------------------------------------------------------------------------------
# Some CUDA-Q calls (for example cudaq.get_state) hold Python's global interpreter lock,
# which would freeze an ordinary Python thread. The sampler therefore runs as a small
# separate process that writes one JSON line per sample to a temporary file. A light
# thread in this notebook reads that file and refreshes the live display.
_SAMPLER_SCRIPT = r'''
import json, sys, time, psutil
pid, interval, path = int(sys.argv[1]), float(sys.argv[2]), sys.argv[3]
handles, pynvml = [], None
try:
    import pynvml
    pynvml.nvmlInit()
    handles = [pynvml.nvmlDeviceGetHandleByIndex(i) for i in range(pynvml.nvmlDeviceGetCount())]
except Exception:
    handles = []
proc = psutil.Process(pid)
with open(path, "a", buffering=1) as out:
    while True:
        try:
            rss = proc.memory_info().rss
        except psutil.Error:
            break
        vm = psutil.virtual_memory()
        gpu = []
        for h in handles:
            try:
                m = pynvml.nvmlDeviceGetMemoryInfo(h)
                gpu.append([m.used, m.total])
            except Exception:
                gpu.append([None, None])
        out.write(json.dumps({"t": time.time(), "rss": rss, "sys_used": vm.total - vm.available,
                              "sys_total": vm.total, "gpu": gpu}) + "\n")
        time.sleep(interval)
'''


class MemoryMonitor:
    """Samples host RAM (psutil) and per-GPU memory (NVML) while code runs.

    Usage:
        monitor = MemoryMonitor(interval_s=0.25, live=True)
        monitor.start()
        ... run quantum code ...
        summary = monitor.stop()
    """

    def __init__(self, interval_s=0.25, live=True, refresh_s=1.0):
        self.interval_s = max(0.05, float(interval_s))     # sampling period of the sampler process
        self.live = live                                    # refresh a one-line live display?
        self.refresh_s = max(0.2, float(refresh_s))         # display refresh period
        self.host = socket.gethostname()                    # node name (matters for multi-node runs)
        self.rank, self.nranks = mpi_rank_info()
        self.samples = []
        self.gpu_names = []
        gpus, self.gpu_status = _nvml_gpu_memory()
        if gpus:
            self.gpu_names = [name for _, _, name in gpus]
        self._proc = psutil.Process(os.getpid())

    def _sample_here(self):
        """One sample taken inside this process (used for baseline, final and fallback)."""
        vm = psutil.virtual_memory()
        gpus, _ = _nvml_gpu_memory()
        return {"t": time.time(), "rss": self._proc.memory_info().rss,
                "sys_used": vm.total - vm.available, "sys_total": vm.total,
                "gpu": [[u, t] for u, t, _ in gpus] if gpus else []}

    def _read_file_samples(self):
        try:
            with open(self._path) as fh:
                lines = fh.readlines()
        except OSError:
            return []
        parsed = []
        for line in lines:
            try:
                parsed.append(json.loads(line))
            except ValueError:                                # partially written last line
                pass
        return parsed

    def _format(self, sample, peak_rss, peak_sys):
        gb = 1e9
        text = (f"[{self.host} rank {self.rank}] t={sample['t'] - self._t0:6.1f}s | "
                f"RAM used {sample['sys_used'] / gb:6.2f}/{sample['sys_total'] / gb:.1f} GB "
                f"({100 * sample['sys_used'] / sample['sys_total']:4.1f}%, peak {peak_sys / gb:.2f}) | "
                f"process RSS {sample['rss'] / gb:5.2f} GB (peak {peak_rss / gb:.2f})")
        for i, (used, total) in enumerate(sample.get("gpu", [])):
            if used is not None:
                text += f" | GPU{i} {used / gb:5.2f}/{total / gb:.0f} GB ({100 * used / total:4.1f}%)"
        return text

    def _display_loop(self):
        while not self._stop_event.wait(self.refresh_s):
            samples = self._read_file_samples() if self._path else []
            if not samples:
                continue
            peak_rss = max(s["rss"] for s in samples)
            peak_sys = max(s["sys_used"] for s in samples)
            self._show(self._format(samples[-1], peak_rss, peak_sys))

    def _show(self, text):
        if self._handle is not None:
            self._handle.update(Pretty(text))               # update the same output line in Jupyter
        else:
            print(text, flush=True)                          # plain scripts: print a new line

    def start(self):
        self._t0 = time.time()
        self._stop_event = threading.Event()
        self._baseline = self._sample_here()
        fd, self._path = tempfile.mkstemp(prefix="cudaq_memmon_", suffix=".jsonl")
        os.close(fd)
        try:                                                 # preferred: independent sampler process
            self._sampler = subprocess.Popen(
                [sys.executable, "-c", _SAMPLER_SCRIPT, str(os.getpid()), str(self.interval_s), self._path],
                stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
            self.mode = "separate sampler process"
            deadline = time.time() + 5.0                     # wait for the first sample (<= 5 s)
            while time.time() < deadline and not self._read_file_samples():
                time.sleep(0.02)
        except Exception as exc:                              # fallback: in-process thread
            self._sampler = None
            self.mode = f"in-process thread (sampler process failed: {exc})"
            self._thread_sampler = threading.Thread(target=self._fallback_loop, daemon=True)
            self._thread_sampler.start()
        self._handle = None
        if self.live:
            if _IN_JUPYTER:
                self._handle = display(Pretty("starting live memory monitor ..."), display_id=True)
            self._display_thread = threading.Thread(target=self._display_loop, daemon=True)
            self._display_thread.start()
        return self

    def _fallback_loop(self):
        with open(self._path, "a", buffering=1) as out:
            while not self._stop_event.wait(self.interval_s):
                out.write(json.dumps(self._sample_here()) + "\n")

    def stop(self):
        final = self._sample_here()
        self._stop_event.set()
        if self._sampler is not None:
            self._sampler.terminate()
            try:
                self._sampler.wait(timeout=5)
            except subprocess.TimeoutExpired:
                self._sampler.kill()
        if self.live:
            self._display_thread.join(timeout=5)
        self.samples = [self._baseline] + self._read_file_samples() + [final]
        self.samples.sort(key=lambda s: s["t"])
        try:
            os.remove(self._path)
        except OSError:
            pass
        summary = self.summary()
        if self.live:
            self._show(self._format(final, summary["rss_peak_gb"] * 1e9, summary["ram_used_peak_gb"] * 1e9)
                       + "  <- monitoring finished")
        return summary

    def summary(self):
        """Baseline / peak values in GB and % for RAM, process RSS and every GPU."""
        gb = 1e9
        base = self.samples[0]
        result = {
            "node": self.host, "mpi_rank": self.rank, "mpi_ranks": self.nranks,
            "sampler": self.mode, "num_samples": len(self.samples),
            "duration_s": self.samples[-1]["t"] - self.samples[0]["t"],
            "ram_total_gb": base["sys_total"] / gb,
            "ram_used_baseline_gb": base["sys_used"] / gb,
            "ram_used_peak_gb": max(s["sys_used"] for s in self.samples) / gb,
            "rss_baseline_gb": base["rss"] / gb,
            "rss_peak_gb": max(s["rss"] for s in self.samples) / gb,
            "gpu_status": self.gpu_status, "gpus": [],
        }
        result["ram_used_peak_pct"] = 100 * result["ram_used_peak_gb"] / result["ram_total_gb"]
        num_gpus = max((len(s.get("gpu", [])) for s in self.samples), default=0)
        for i in range(num_gpus):
            used = [s["gpu"][i][0] for s in self.samples if len(s.get("gpu", [])) > i and s["gpu"][i][0] is not None]
            totals = [s["gpu"][i][1] for s in self.samples if len(s.get("gpu", [])) > i and s["gpu"][i][1]]
            if not used or not totals:
                continue
            result["gpus"].append({
                "index": i, "name": self.gpu_names[i] if i < len(self.gpu_names) else f"GPU{i}",
                "total_gb": totals[0] / gb, "used_baseline_gb": used[0] / gb, "used_peak_gb": max(used) / gb,
                "used_peak_pct": 100 * max(used) / totals[0]})
        return result

    def plot(self, ax=None, title="Memory during execution"):
        """Plot RAM, process RSS and per-GPU memory versus time."""
        if ax is None:
            _, ax = plt.subplots(figsize=(9, 3.2))
        t = np.array([s["t"] for s in self.samples]) - self.samples[0]["t"]
        ax.plot(t, [s["sys_used"] / 1e9 for s in self.samples], label="host RAM used (all processes)")
        ax.plot(t, [s["rss"] / 1e9 for s in self.samples], label="this notebook process (RSS)")
        num_gpus = max((len(s.get("gpu", [])) for s in self.samples), default=0)
        for i in range(num_gpus):
            pts = [(tt, s["gpu"][i][0] / 1e9) for tt, s in zip(t, self.samples)
                   if len(s.get("gpu", [])) > i and s["gpu"][i][0] is not None]
            if pts:
                ax.plot(*zip(*pts), label=f"GPU{i} memory used")
        ax.set_ylim(bottom=0)                                 # start at zero: do not exaggerate small changes
        ax.set_xlabel("time since start of execution [s]")
        ax.set_ylabel("memory [GB]")
        ax.set_title(title)
        ax.legend(fontsize=8, loc="best")
        ax.grid(alpha=0.3)
        return ax


def print_memory_summary(mem):
    """Print the memory section of the benchmark (per node and per GPU)."""
    print(f"Node {mem['node']} (MPI rank {mem['mpi_rank']}/{mem['mpi_ranks']}), sampler: {mem['sampler']}, "
          f"{mem['num_samples']} samples over {mem['duration_s']:.1f} s")
    print(f"  Host RAM used : baseline {mem['ram_used_baseline_gb']:.2f} GB -> peak {mem['ram_used_peak_gb']:.2f} GB "
          f"of {mem['ram_total_gb']:.1f} GB ({mem['ram_used_peak_pct']:.1f}%)")
    print(f"  Process RSS   : baseline {mem['rss_baseline_gb']:.2f} GB -> peak {mem['rss_peak_gb']:.2f} GB "
          f"(increase {mem['rss_peak_gb'] - mem['rss_baseline_gb']:+.2f} GB)")
    if mem["gpus"]:
        for g in mem["gpus"]:
            print(f"  GPU{g['index']} ({g['name']}): baseline {g['used_baseline_gb']:.2f} GB -> peak "
                  f"{g['used_peak_gb']:.2f} GB of {g['total_gb']:.1f} GB ({g['used_peak_pct']:.1f}%)")
    else:
        print(f"  GPU memory    : not monitored - {mem['gpu_status']}")
    if mem["mpi_ranks"] > 1:
        print("  (multi-node/multi-rank run: every MPI rank prints its own node's line)")


# ---------------------------------------------------------------------------------
# 4. Timing: construction vs compilation vs execution
# ---------------------------------------------------------------------------------
# CUDA-Q (>= 0.16) contains a span tracer that records how long each internal phase
# takes. We use it for the FIRST launch of a kernel only:
#   construction = Python source -> Quake IR      (span "PyKernelDecorator.compile")
#   compilation  = MLIR passes + LLVM JIT          (span "Compiler::runPassPipeline")
#   execution    = running the simulation/QPU job  (spans "streamlinedLaunchModule" and
#                  "...finalizeExecutionContext", where get_state() flushes queued gates)
# The tracer adds ~1 microsecond per gate, so later launches are timed without it.
_EXECUTION_SPANS = {"streamlinedLaunchModule", "DefaultPlatform::finalizeExecutionContext",
                    "BasicExecutionManager::finalizeExecutionContext"}


def _span_union_seconds(events, names):
    """Total time covered by spans with the given names (nested/overlapping spans counted once)."""
    intervals = sorted((e["ts"], e["ts"] + e["dur"]) for e in events if e.get("name") in names and "dur" in e)
    total, start, end = 0.0, None, None
    for s, e in intervals:
        if end is None or s > end:                            # a new, separate interval begins
            if end is not None:
                total += end - start
            start, end = s, e
        else:                                                 # overlapping/nested: extend
            end = max(end, e)
    if end is not None:
        total += end - start
    return total / 1e6                                        # tracer timestamps are microseconds


def profile_first_launch(launch):
    """Run launch() once under the CUDA-Q tracer. Returns (result, phase-times dict)."""
    tracer, backend_obj = None, None
    try:
        from cudaq.util import trace as tracer
        backend_obj = tracer.ChromeBackend()                  # in-memory event recorder
        tracer.set_backend(backend_obj)
    except Exception:                                         # older CUDA-Q: no tracer
        tracer = None
    t_start = time.perf_counter()
    try:
        result = launch()
    finally:
        wall = time.perf_counter() - t_start
        if tracer is not None:
            tracer.reset_backend()                            # stop recording
    phases = {"wall_s": wall, "construction_s": None, "compilation_s": None, "execution_s": None,
              "method": "wall clock only (tracer not available)"}
    if backend_obj is not None:
        events = json.loads(backend_obj.to_json()).get("traceEvents", [])
        construct = _span_union_seconds(events, {"cudaq.kernel.kernel_decorator.PyKernelDecorator.compile"})
        compile_ = _span_union_seconds(events, {"Compiler::runPassPipeline"})
        execute = _span_union_seconds(events, _EXECUTION_SPANS)
        if compile_ > 0 or execute > 0:
            phases.update(construction_s=construct, compilation_s=compile_, execution_s=execute,
                          method="CUDA-Q span tracer (first launch)")
        else:
            phases["method"] = "wall clock only (this target does not report compile/execute spans)"
    return result, phases


def fmt_seconds(value):
    return "not measured" if value is None else f"{value:.4f} s"


# ---------------------------------------------------------------------------------
# 5. Execution-time estimate: measure the cost of one gate on the active backend
# ---------------------------------------------------------------------------------
@cudaq.kernel
def _calibration_kernel(n: int, repetitions: int):
    qubits = cudaq.qvector(n)
    for _ in range(repetitions):
        for i in range(n):
            ry(0.1, qubits[i])
    mz(qubits)


def measure_gate_time(cfg, n_qubits):
    """Seconds per gate at n_qubits, measured on the active target (rough, +-3x).

    Two short runs with different gate counts are timed; their difference removes fixed
    overheads. If n_qubits is large, the calibration runs on fewer qubits and is scaled by
    2^dN (state vector) or 4^dN (density matrix) because cost grows with the state size.
    """
    method = cfg["method"]
    if cfg["device"] == "hardware":
        return None, "not estimated: hardware time is dominated by queueing and provider limits"
    if method in ("mps", "tensornet", "stabilizer"):
        return None, f"not estimated: '{method}' cost depends on entanglement/circuit structure"
    cap = 9 if method == "density_matrix" else (14 if cfg["device"] == "cpu" else 24)
    n_cal = max(2, min(n_qubits, cap))
    def timed(repetitions):
        start = time.perf_counter()
        cudaq.sample(_calibration_kernel, n_cal, repetitions, shots_count=1)
        return time.perf_counter() - start
    timed(1)                                                  # warm-up: JIT compilation
    r1, r2 = 2, 12
    t1 = min(timed(r1) for _ in range(2))
    t2 = min(timed(r2) for _ in range(2))
    per_gate = max((t2 - t1) / ((r2 - r1) * n_cal), 1e-9)
    growth = 4 if method == "density_matrix" else 2
    scaled = per_gate * growth ** (n_qubits - n_cal)
    detail = f"measured {per_gate * 1e3:.4f} ms/gate at {n_cal} qubits"
    if n_qubits > n_cal:
        detail += f", extrapolated x{growth}^{n_qubits - n_cal} to {n_qubits} qubits"
    return scaled, detail


def format_duration(seconds):
    if seconds is None:
        return "not estimated"
    if seconds < 1:
        return f"{seconds * 1e3:.0f} ms"
    if seconds < 120:
        return f"{seconds:.1f} s"
    if seconds < 7200:
        return f"{seconds / 60:.1f} min"
    if seconds < 3 * 86400:
        return f"{seconds / 3600:.1f} h"
    return f"{seconds / 86400:.1e} days"


# ---------------------------------------------------------------------------------
# 6. Circuit drawing, statistics and export
# ---------------------------------------------------------------------------------
def circuit_statistics(kernel, *args):
    """Gate counts and depth of the FULL circuit via cudaq.estimate_resources (no simulation)."""
    try:
        res = cudaq.estimate_resources(kernel, *args)
        by_type = dict(res.to_dict())
        measurements = sum(v for k, v in by_type.items() if k in ("mz", "mx", "my"))
        return {"qubits": res.num_qubits, "gates (excluding measurements)": res.count() - measurements,
                "measurements": measurements, "depth": res.depth,
                "multi-qubit gates": res.multi_qubit_gate_count, "gate counts by type": by_type}
    except Exception as exc:
        return {"error": f"estimate_resources failed: {type(exc).__name__}: {exc}"}


def _render_text_png(text, path):
    lines = text.splitlines() or [""]
    width, height = max(len(line) for line in lines), len(lines)
    fig_w, fig_h = max(4.0, 0.075 * width), max(1.0, 0.19 * height)
    dpi = int(min(150, 30000 / (fig_w * 1.0)))               # keep huge drawings below ~30k pixels
    fig = plt.figure(figsize=(fig_w, fig_h))
    fig.text(0.0, 1.0, text, family="DejaVu Sans Mono", fontsize=9, va="top", ha="left")
    fig.savefig(path, dpi=max(dpi, 30), bbox_inches="tight", facecolor="white")
    plt.close(fig)


def show_circuit(kernel, args, n_shown, n_full, title, export=False, formats=("txt", "png", "qir"),
                 out_dir="exports", basename="circuit", full_kernel=None, full_args=()):
    """Print the circuit drawing of `kernel(*args)` (built with n_shown <= 10 qubits) and export it.

    Export formats:
      "txt"  - the drawing as UTF-8 text          "png"  - the drawing as an image
      "qir"  - the FULL N-qubit circuit (full_kernel(*full_args)) as QIR, the LLVM-based
               intermediate representation CUDA-Q sends to simulators and QPUs
      "qasm" - the FULL circuit as OpenQASM 2.0 (only for circuits CUDA-Q can unroll into
               plain gates; otherwise skipped with an explanation)
    """
    drawing = cudaq.draw(kernel, *args)
    if n_full > n_shown:
        banner = (f"CIRCUIT VISUALIZATION EXAMPLE: {n_shown}-qubit version of the circuit. "
                  f"This is NOT the complete {n_full}-qubit circuit that will be executed.")
    else:
        banner = f"Circuit drawing ({n_shown} qubits). {title}"
    print("=" * min(len(banner), 110))
    print(banner)
    print("=" * min(len(banner), 110))
    print(drawing)
    saved = []
    if export:
        os.makedirs(out_dir, exist_ok=True)
        stem = os.path.join(out_dir, f"{basename}_{n_shown}q")
        if "txt" in formats:
            with open(stem + ".txt", "w", encoding="utf-8") as fh:
                fh.write(banner + "\n\n" + drawing)
            saved.append(stem + ".txt")
        if "png" in formats:
            try:
                _render_text_png(banner + "\n\n" + drawing, stem + ".png")
                saved.append(stem + ".png")
            except Exception as exc:
                print(f"PNG export skipped: {exc}")
        if "qir" in formats and full_kernel is not None:
            try:
                qir = cudaq.translate(full_kernel, *full_args, format="qir")
                if "translation failed" in qir:
                    raise RuntimeError("CUDA-Q could not translate this circuit to QIR")
                path = os.path.join(out_dir, f"{basename}_{n_full}q_full.ll")
                with open(path, "w") as fh:
                    fh.write(qir)
                saved.append(path)
            except Exception as exc:
                print(f"QIR export skipped: {type(exc).__name__}: {exc}")
        if "qasm" in formats and full_kernel is not None:
            try:
                specialised = cudaq.synthesize(full_kernel, *full_args) if full_args else full_kernel
                qasm = cudaq.translate(specialised, format="openqasm2")
                if "translation failed" in qasm:
                    raise RuntimeError("translation failed")
                path = os.path.join(out_dir, f"{basename}_{n_full}q_full.qasm")
                with open(path, "w") as fh:
                    fh.write(qasm)
                saved.append(path)
            except Exception as exc:
                print("OpenQASM 2.0 export skipped: CUDA-Q 0.16 can only emit OpenQASM for circuits that "
                      f"unroll into plain gates without sub-kernels or list arguments ({type(exc).__name__}). "
                      "Use the 'qir' export instead.")
        for path in saved:
            print(f"Exported: {path}")
    return drawing


# ---------------------------------------------------------------------------------
# 7. Final benchmark report
# ---------------------------------------------------------------------------------
def build_report(algorithm, cfg, n_qubits, shots, phases, mem, extra_times=None):
    """Collect the standard benchmark fields into an ordered dict."""
    target = cudaq.get_target()
    report = {
        "Algorithm": algorithm,
        "Date (UTC)": datetime.datetime.now(datetime.timezone.utc).strftime("%Y-%m-%d %H:%M:%S"),
        "CUDA-Q version": CUDAQ_VERSION,
        "Number of qubits": n_qubits,
        "Backend (requested -> used)": f"{cfg['backend_requested']} -> {cfg['device']}",
        "CUDA-Q target": target.name + (f" [{cfg['kwargs']}]" if cfg["kwargs"] else ""),
        "Simulation method": cfg["method"],
        "Numerical precision": cfg["precision"],
        "Number of shots": shots,
        "Circuit construction time": fmt_seconds(phases.get("construction_s")),
        "Compilation time (MLIR + JIT)": fmt_seconds(phases.get("compilation_s")),
        "Quantum execution time": fmt_seconds(phases.get("execution_s")),
    }
    for key, value in (extra_times or {}).items():
        report[key] = fmt_seconds(value)
    report["Total wall time of execution cell"] = fmt_seconds(phases.get("total_wall_s"))
    report["Timing method"] = phases.get("method", "")
    report["Peak host RAM used"] = f"{mem['ram_used_peak_gb']:.2f} GB ({mem['ram_used_peak_pct']:.1f}% of {mem['ram_total_gb']:.1f} GB)"
    report["Peak notebook process RSS"] = f"{mem['rss_peak_gb']:.2f} GB (baseline {mem['rss_baseline_gb']:.2f} GB)"
    if mem["gpus"]:
        for g in mem["gpus"]:
            report[f"Peak GPU{g['index']} memory ({g['name']})"] = (f"{g['used_peak_gb']:.2f} GB "
                                                                   f"({g['used_peak_pct']:.1f}% of {g['total_gb']:.1f} GB)")
    else:
        report["Peak GPU memory"] = f"not available - {mem['gpu_status']}"
    report["Node / MPI rank"] = f"{mem['node']} / rank {mem['mpi_rank']} of {mem['mpi_ranks']}"
    return report


def show_report(report, save=True, out_dir="exports", name="benchmark_report", figure=None):
    """Pretty-print the report and optionally save it as JSON (+ figure as PNG)."""
    width = max(len(str(k)) for k in report)
    print("=" * 100)
    print(f"{'FINAL BENCHMARK REPORT':^100}")
    print("=" * 100)
    for key, value in report.items():
        print(f"{str(key):<{width}} : {value}")
    print("=" * 100)
    if save:
        os.makedirs(out_dir, exist_ok=True)
        json_path = os.path.join(out_dir, f"{name}.json")
        with open(json_path, "w") as fh:
            json.dump({str(k): (v if isinstance(v, (int, float, str, bool, type(None))) else str(v))
                       for k, v in report.items()}, fh, indent=2)
        print(f"Report saved to {json_path}")
        if figure is not None:
            fig_path = os.path.join(out_dir, f"{name}.png")
            figure.savefig(fig_path, dpi=120, bbox_inches="tight")
            print(f"Final figure saved to {fig_path}")


def bit_reverse_order(vector, n_qubits):
    """Convert between CUDA-Q's state-vector order and 'qubit 0 = most significant bit'.

    cudaq.get_state() stores amplitude index i = sum_q b_q * 2^q (qubit 0 is the LEAST
    significant bit), while measured bitstrings are printed with qubit 0 FIRST (leftmost).
    Reversing the qubit axes converts one convention into the other (and back).
    """
    vector = np.asarray(vector)
    return vector.reshape([2] * n_qubits).transpose(list(range(n_qubits))[::-1]).reshape(-1)


print("Benchmark utilities loaded.")

In [ ]:
# ---------------------------------------------------------------------------------
# Activate the selected backend and print what is available on this machine
# ---------------------------------------------------------------------------------
ALLOWED_METHODS = ["statevector", "mps", "tensornet", "density_matrix"]   # simulation methods that make sense for this notebook
np.random.seed(SEED)                          # NumPy random numbers are reproducible too

backend_cfg = resolve_backend(BACKEND, SIMULATION_METHOD, GPU_PRECISION, ALLOWED_METHODS,
                              HARDWARE_PROVIDER, HARDWARE_EMULATE, HARDWARE_MACHINE)
active_target = initialize_backend(backend_cfg, seed=SEED)   # calls cudaq.set_target(...)

print(f"Requested backend : {BACKEND}  ->  device: {backend_cfg['device']}")
print(f"CUDA-Q target     : {active_target.name}  ({backend_cfg['description']})")
print(f"Simulation method : {backend_cfg['method']},  precision: {backend_cfg['precision']}")
print(f"GPUs visible      : {backend_cfg['num_gpus']},  CPU cores: {os.cpu_count()},  "
      f"host RAM: {psutil.virtual_memory().total / 1e9:.1f} GB")
gpu_list, gpu_status = _nvml_gpu_memory()
for i, (used, total, name) in enumerate(gpu_list or []):
    print(f"  GPU{i}: {name}, {total / 1e9:.1f} GB total, {used / 1e9:.2f} GB in use")
if not gpu_list:
    print(f"  GPU monitoring: {gpu_status}")

objective_shots = OBJECTIVE_SHOTS if OBJECTIVE_SHOTS is not None else (SHOTS if backend_cfg["device"] == "hardware" else None)
print(f"Energy evaluation : {'exact expectation value' if objective_shots is None else f'{objective_shots} shots per term group'}")

---
## Section 6 - Expected output (before execution)

**Expected graphs (Section 9)**
1. *Energy convergence* - $E(\boldsymbol\theta)$ per optimizer evaluation, decreasing towards the exact $E_0$ and
   never below it (variational principle; only shot noise can push single evaluations slightly below).
2. *Error to the reference* on a logarithmic axis.
3. *Dominant bitstrings of the optimised state* and the *memory timeline*.

**Expected numbers:** the reference energy $E_0$ of Section 4. The final VQE energy should be slightly above it; the
relative error depends on the ansatz depth and optimizer budget (for H$_2$ with the UCC ansatz: below 1.6 mHa).

**Execution time and memory** are computed below (time = calibrated estimate).

In [ ]:
# ---------------------------------------------------------------------------------
# Section 6 - Memory requirement and execution-time estimate
# ---------------------------------------------------------------------------------
state_bytes = estimate_memory_bytes(n_qubits, backend_cfg["method"], backend_cfg["precision"])
print("Expected memory")
check_memory(state_bytes, backend_cfg, MEMORY_SAFETY_FRACTION)

if ansatz == "ucc":
    gates_per_circuit = 2 + 20                          # 2 X + exp_pauli on 4 qubits (~20 gates after decomposition)
else:
    gates_per_circuit = num_parameters + int(ANSATZ_LAYERS) * (n_qubits - 1)   # RY rotations + CNOT ladders
seconds_per_gate, timing_note = measure_gate_time(backend_cfg, n_qubits)
print("\nExpected execution time (ESTIMATE)")
print(f"  Gates per ansatz circuit : ~{gates_per_circuit:,}")
print(f"  Energy evaluations       : up to {MAX_EVALUATIONS} (+1 final sampling run)")
print(f"  Calibration              : {timing_note}")
if seconds_per_gate is not None:
    # computing <H> costs about one extra state pass per group of commuting terms on simulators: x2 is a rough allowance
    estimate = (MAX_EVALUATIONS + 1) * gates_per_circuit * seconds_per_gate * 2
    print(f"  Estimated total          : ~{format_duration(estimate)} (rough: energy evaluation adds ~1x the circuit cost)")
if reference_energy is not None:
    print(f"\nTarget: E0 = {reference_energy:.8f}; the VQE energy must stay >= E0.")

---
## Section 7 - Quantum circuit generation

* `hea_ansatz(n, layers, theta)` - hardware-efficient ansatz: `layers` x [ $R_Y$ on all qubits, CNOT ladder ], final $R_Y$ layer.
* `ucc_ansatz(theta)` - H$_2$: $X$ on qubits 0 and 1 (Hartree-Fock $|1100\rangle$), then `exp_pauli(theta, "XXXY")` = $e^{i\theta X_0X_1X_2Y_3}$.
* `*_sampled` versions add a measurement, used once at the end to show the dominant bitstrings.

In [ ]:
# ---------------------------------------------------------------------------------
# Section 7a - CUDA-Q kernels (ansatz circuits)
# ---------------------------------------------------------------------------------
def build_vqe_kernels():
    """Create and return fresh CUDA-Q kernels (called again in the execution cell)."""

    @cudaq.kernel
    def hea_ansatz(n: int, layers: int, theta: list[float]):
        """Hardware-efficient ansatz with (layers + 1) * n rotation angles."""
        qubits = cudaq.qvector(n)
        k = 0                                              # index of the next angle in theta
        for layer in range(layers):
            for q in range(n):
                ry(theta[k], qubits[q])                   # single-qubit rotation: amplitude between |0> and |1>
                k += 1
            for q in range(n - 1):
                x.ctrl(qubits[q], qubits[q + 1])          # CNOT ladder: entangles neighbouring qubits
        for q in range(n):
            ry(theta[k], qubits[q])                       # final rotation layer
            k += 1

    @cudaq.kernel
    def hea_ansatz_sampled(n: int, layers: int, theta: list[float]):
        """Same circuit + measurement of every qubit."""
        qubits = cudaq.qvector(n)
        k = 0
        for layer in range(layers):
            for q in range(n):
                ry(theta[k], qubits[q])
                k += 1
            for q in range(n - 1):
                x.ctrl(qubits[q], qubits[q + 1])
        for q in range(n):
            ry(theta[k], qubits[q])
            k += 1
        mz(qubits)

    @cudaq.kernel
    def ucc_ansatz(theta: list[float]):
        """H2: Hartree-Fock |1100> followed by the double excitation exp(i theta X0 X1 X2 Y3)."""
        qubits = cudaq.qvector(4)
        x(qubits[0])                                       # occupy spin-orbital 0 ...
        x(qubits[1])                                       # ... and 1 (two electrons in the bonding orbital)
        exp_pauli(theta[0], qubits, "XXXY")                # cos(theta)|1100> - sin(theta)|0011>

    @cudaq.kernel
    def ucc_ansatz_sampled(theta: list[float]):
        qubits = cudaq.qvector(4)
        x(qubits[0])
        x(qubits[1])
        exp_pauli(theta[0], qubits, "XXXY")
        mz(qubits)

    return {"hea": hea_ansatz, "hea_sampled": hea_ansatz_sampled, "ucc": ucc_ansatz, "ucc_sampled": ucc_ansatz_sampled}


def ansatz_call(kernels, theta, sampled=False):
    """(kernel, arguments) of the configured ansatz for angle vector theta."""
    if ansatz == "ucc":
        return kernels["ucc_sampled" if sampled else "ucc"], (list(theta),)
    return kernels["hea_sampled" if sampled else "hea"], (n_qubits, int(ANSATZ_LAYERS), list(theta))


init_rng = np.random.default_rng(SEED)
initial_theta = np.zeros(1) if ansatz == "ucc" else init_rng.normal(0.0, INITIAL_PARAMETER_SCALE, num_parameters)
kernels_vis = build_vqe_kernels()
print(f"Initial parameters: {num_parameters} values, first ones {np.round(initial_theta[:5], 4).tolist()}")

In [ ]:
# ---------------------------------------------------------------------------------
# Section 7b - Circuit visualization (at most 10 qubits) and full-circuit statistics
# ---------------------------------------------------------------------------------
n_draw = min(int(DRAW_QUBITS), 10, n_qubits)
if not 0 < n_draw <= 10:
    raise ValueError("DRAW_QUBITS must be between 1 and 10.")
full_kernel, full_args = ansatz_call(kernels_vis, initial_theta, sampled=True)
if ansatz == "ucc":
    draw_kernel, draw_args = full_kernel, full_args
else:
    draw_layers = min(int(ANSATZ_LAYERS), 2)                         # at most 2 layers keep the drawing readable
    draw_kernel = kernels_vis["hea_sampled"]
    draw_args = (n_draw, draw_layers, [0.1] * ((draw_layers + 1) * n_draw))
    print(f"The drawing shows {draw_layers} ansatz layer(s) on {n_draw} qubits (angles shown as 0.1).")
show_circuit(draw_kernel, draw_args, n_draw if ansatz != "ucc" else 4, n_qubits, title="Ansatz circuit.",
             export=EXPORT_CIRCUIT, formats=EXPORT_FORMATS, out_dir=EXPORT_DIR, basename="vqe_ansatz",
             full_kernel=full_kernel, full_args=full_args)
stats = circuit_statistics(full_kernel, *full_args)
print(f"\nFULL ansatz statistics ({n_qubits} qubits):")
for key, value in stats.items():
    print(f"  {key:<32}: {value}")

---
## Section 8 - Execution with live memory monitoring

Running the next cell:
1. initialises the selected backend,
2. builds the ansatz circuit (fresh kernels),
3. runs the VQE loop - every evaluation is one `cudaq.observe(ansatz, H, theta)` - and finally samples the optimised state,
4. monitors host RAM and GPU memory during the whole cell,
5. prints a short summary.

**How the timings are measured**

| Phase | What happens | How it is measured |
|---|---|---|
| Circuit construction | the Python kernel is translated into CUDA-Q's quantum IR (Quake/MLIR) | CUDA-Q tracer span on the first launch |
| Compilation | MLIR optimisation passes + LLVM just-in-time compilation to machine code | CUDA-Q tracer span on the first launch (cached afterwards) |
| Quantum execution | the simulator (or QPU) applies the gates and measures | tracer span on the first launch + wall-clock time of all later launches |

**How memory is monitored:** a separate lightweight process samples memory every `MEMORY_SAMPLE_INTERVAL_S`
seconds - host RAM used by all processes and this notebook's resident memory (RSS) with `psutil`, and the
memory of every NVIDIA GPU with NVML (device-wide, so it includes other processes on that GPU). Some CUDA-Q
calls (e.g. `cudaq.get_state`) hold Python's interpreter lock, so the *live line* may pause during them,
but the background sampler keeps recording, so the reported peaks are still captured. Peaks shorter than
the sampling interval can be missed. Nothing is estimated or invented: if a monitor is unavailable (e.g. no
NVIDIA driver), the report says so.

In [ ]:
# =====================================================================================
# >>> EXECUTION CELL - VQE optimisation with live memory monitoring <<<
# =====================================================================================
execution_ok, execution_error = False, None
energy_history, theta_history = [], []
phases, later_launch_time = None, 0.0
monitor = MemoryMonitor(MEMORY_SAMPLE_INTERVAL_S, live=LIVE_MEMORY_DISPLAY)
monitor.start()
cell_start = time.perf_counter()
try:
    # Step 1 - initialise the backend
    initialize_backend(backend_cfg, seed=SEED)
    # Step 2 - build the circuit (fresh kernels)
    kernels = build_vqe_kernels()
    observe_kwargs = {} if objective_shots is None else {"shots_count": int(objective_shots)}

    def energy(theta):
        """E(theta) = <psi(theta)|H|psi(theta)>; the first launch is traced for the timing breakdown."""
        global phases, later_launch_time
        kernel, args = ansatz_call(kernels, theta)
        launch = lambda: cudaq.observe(kernel, hamiltonian, *args, **observe_kwargs)
        if phases is None:
            result, phases = profile_first_launch(launch)
        else:
            t0 = time.perf_counter()
            result = launch()
            later_launch_time += time.perf_counter() - t0
        value = result.expectation()
        energy_history.append(value)
        theta_history.append(np.array(theta, dtype=float))
        return value

    # Step 3 - variational optimisation
    method = OPTIMIZER.strip()
    options = {"COBYLA": {"maxiter": MAX_EVALUATIONS, "rhobeg": 0.3}, "NELDER-MEAD": {"maxfev": MAX_EVALUATIONS},
               "POWELL": {"maxfev": MAX_EVALUATIONS}, "L-BFGS-B": {"maxfun": MAX_EVALUATIONS}}.get(method.upper())
    if options is None:
        raise ValueError("OPTIMIZER must be 'COBYLA', 'Nelder-Mead', 'Powell' or 'L-BFGS-B'.")
    optimisation_start = time.perf_counter()
    opt = minimize(energy, initial_theta, method=method, options=options)
    optimisation_time = time.perf_counter() - optimisation_start
    best_index = int(np.argmin(energy_history))
    best_theta, vqe_energy = theta_history[best_index], energy_history[best_index]

    # Step 4 - sample the optimised state to see which bitstrings dominate it
    t0 = time.perf_counter()
    kernel_s, args_s = ansatz_call(kernels, best_theta, sampled=True)
    final_counts = cudaq.sample(kernel_s, *args_s, shots_count=SHOTS)
    sampling_time = time.perf_counter() - t0
    execution_ok = True
except Exception as exc:
    execution_error = exc
    print(f"\nEXECUTION FAILED: {type(exc).__name__}: {exc}")
    print("Hints: reduce N_QUBITS or MAX_EVALUATIONS, choose another BACKEND / SIMULATION_METHOD (README).")
finally:
    memory = monitor.stop()
    total_wall = time.perf_counter() - cell_start

if execution_ok:
    phases["total_wall_s"] = total_wall
    print(f"\nOptimizer       : {method}, {len(energy_history)} evaluations ({opt.message})")
    print(f"VQE energy      : {vqe_energy:.10f}")
    if reference_energy is not None:
        abs_error = vqe_energy - reference_energy
        rel_error = abs_error / abs(reference_energy)
        print(f"Reference E0    : {reference_energy:.10f}")
        print(f"Error           : {abs_error:.3e} (relative {rel_error:.3%})")
        if kind == "h2":
            print(f"Chemical accuracy (1.6 mHa): {'REACHED' if abs(abs_error) < 1.6e-3 else 'not reached'}")
    print(f"Timing          : construction {fmt_seconds(phases['construction_s'])}, compilation {fmt_seconds(phases['compilation_s'])}, "
          f"optimisation loop {optimisation_time:.2f} s, final sampling {sampling_time:.2f} s")
    print(f"Total cell wall time: {total_wall:.2f} s\n")
print_memory_summary(memory)

---
## Section 9 - Results and visualization

In [ ]:
# ---------------------------------------------------------------------------------
# Section 9 - Energy convergence, error and dominant bitstrings
# ---------------------------------------------------------------------------------
if not execution_ok:
    print("Skipped: the execution cell failed (see the error message above).")
else:
    evaluations = np.arange(1, len(energy_history) + 1)
    best_so_far = np.minimum.accumulate(energy_history)
    top_states = sorted(final_counts.items(), key=lambda kv: kv[1], reverse=True)[:8]
    print("Dominant bitstrings of the optimised state (qubit 0 first):")
    for bits, c in top_states:
        print(f"  {bits}  {c / SHOTS:.4f}")
    print(f"\nOptimised parameters (first 10): {np.round(best_theta[:10], 4).tolist()}")

    fig = plt.figure(figsize=(12, 7.6))
    grid = fig.add_gridspec(2, 2, height_ratios=[1.1, 0.9], hspace=0.45, wspace=0.3)
    ax_e = fig.add_subplot(grid[0, 0])
    ax_e.plot(evaluations, energy_history, color=AXIS_COLOR, linewidth=1.0, label="energy of each evaluation")
    ax_e.plot(evaluations, best_so_far, color=BLUE, label="best energy so far")
    if reference_energy is not None:
        ax_e.axhline(reference_energy, color=INK_SECONDARY, linewidth=1)
        ax_e.text(evaluations[-1], reference_energy, f"exact E0 = {reference_energy:.5f} ", ha="right", va="top",
                  fontsize=8, color=INK_SECONDARY)                   # below the line: the curve approaches from above
    ax_e.set_xlabel("optimizer evaluation")
    ax_e.set_ylabel("energy")
    ax_e.set_title(f"VQE energy convergence ({kind.upper()}, {n_qubits} qubits)", loc="left")
    ax_e.legend(fontsize=8, loc="upper right")
    ax_r = fig.add_subplot(grid[0, 1])
    if reference_energy is not None:
        ax_r.semilogy(evaluations, np.maximum(best_so_far - reference_energy, 1e-12), color=BLUE)
        if kind == "h2":
            ax_r.axhline(1.6e-3, color=INK_SECONDARY, linewidth=1)
            ax_r.text(evaluations[-1], 1.6e-3, "chemical accuracy 1.6 mHa ", ha="right", va="bottom", fontsize=8,
                      color=INK_SECONDARY)
        ax_r.set_ylabel("E_best - E0 (log scale)")
        ax_r.set_title("Error to the exact ground energy", loc="left")
    else:
        ax_r.text(0.5, 0.5, "no exact reference for this size", ha="center", va="center", transform=ax_r.transAxes)
    ax_r.set_xlabel("optimizer evaluation")
    ax_s = fig.add_subplot(grid[1, 0])
    positions = np.arange(len(top_states))
    ax_s.bar(positions, [c / SHOTS for _, c in top_states], width=0.5, color=BLUE)
    ax_s.set_xticks(positions, [b for b, _ in top_states], rotation=60, fontsize=7, family="DejaVu Sans Mono")
    ax_s.set_ylabel("probability")
    ax_s.set_title(f"Dominant bitstrings of the optimised state ({SHOTS} shots)", loc="left")
    ax_s.grid(axis="x", visible=False)
    ax_m = fig.add_subplot(grid[1, 1])
    monitor.plot(ax=ax_m, title="Memory during the execution cell")
    plt.show()

---
## Section 10 - Final benchmark report

In [ ]:
# ---------------------------------------------------------------------------------
# Section 10 - Final benchmark report
# ---------------------------------------------------------------------------------
if not execution_ok:
    print(f"No report: execution failed with {type(execution_error).__name__}: {execution_error}")
else:
    phases_report = dict(phases)
    phases_report["execution_s"] = phases["execution_s"] + later_launch_time if phases["execution_s"] is not None else None
    report = build_report("VQE (variational quantum eigensolver)", backend_cfg, n_qubits,
                          objective_shots if objective_shots else "exact expectation values (final sampling: %d)" % SHOTS,
                          phases_report, memory,
                          extra_times={"Final sampling run": sampling_time,
                                       "Classical optimizer + Python overhead": optimisation_time - later_launch_time - phases["wall_s"]})
    report["Hamiltonian"] = f"{kind.upper()}: {len(pauli_terms)} Pauli terms" + (f", J={TFIM_J}, h={TFIM_H}" if kind == "tfim" else "")
    report["Ansatz / parameters"] = f"{'UCC double excitation' if ansatz == 'ucc' else f'hardware-efficient, {ANSATZ_LAYERS} layers'} / {num_parameters}"
    report["Optimizer / evaluations"] = f"{OPTIMIZER} / {len(energy_history)}"
    report["Expected result"] = (f"E0 = {reference_energy:.10f} ({reference_kind.split(';')[0]})" if reference_energy is not None
                                 else "no exact reference (system too large)")
    report["Actual result"] = f"E_VQE = {vqe_energy:.10f}"
    if reference_energy is not None:
        report["Error E_VQE - E0"] = f"{vqe_energy - reference_energy:.3e} (relative {(vqe_energy - reference_energy) / abs(reference_energy):.4%})"
        if kind == "h2":
            report["Chemical accuracy (1.6 mHa)"] = "REACHED" if abs(vqe_energy - reference_energy) < 1.6e-3 else "not reached"
    final_fig, ax = plt.subplots(figsize=(9, 3.4))
    ax.plot(np.arange(1, len(energy_history) + 1), np.minimum.accumulate(energy_history), color=BLUE, label="VQE (best so far)")
    if reference_energy is not None:
        ax.axhline(reference_energy, color=INK_SECONDARY, linewidth=1, label="exact E0")
    ax.set_xlabel("optimizer evaluation")
    ax.set_ylabel("energy")
    ax.set_title(f"VQE {kind.upper()} N={n_qubits} on {cudaq.get_target().name}", loc="left")
    ax.legend(fontsize=8)
    final_fig.tight_layout()
    show_report(report, save=SAVE_REPORT, out_dir=EXPORT_DIR, name="vqe_benchmark_report", figure=final_fig)
    plt.show()

---
### Things to try next
* `HAMILTONIAN = "H2"`: chemistry example with the UCC ansatz - it reaches chemical accuracy in ~20 evaluations.
* `TFIM_H = 1.0` (critical point): the 2-layer ansatz stalls a few percent above $E_0$; raise `ANSATZ_LAYERS` and
  `MAX_EVALUATIONS` and watch the error plot.
* `OBJECTIVE_SHOTS = 2000`: optimise with shot noise, as on hardware - the energy curve becomes noisy.
* `N_QUBITS = 24` with `BACKEND = "gpu"`: each evaluation is a full state-vector simulation; compare the
  reported execution times between CPU and GPU.